# Phân Cụm Khách Hàng Bán Lẻ Trực Tuyến bằng Agglomerative Hierarchical Clustering

## 📌 Giới Thiệu Đồ Án
Trong bài toán phân khúc khách hàng bán lẻ trực tuyến (**Online Retail Customer Segmentation**), mô hình hóa hành vi khách hàng qua 3 chỉ số then chốt **RFM**:
- **Recency ($R$)**: Số ngày trôi qua kể từ lần mua sắm gần nhất của khách hàng (càng nhỏ càng tốt).
- **Frequency ($F$)**: Tổng số lượng hóa đơn/đơn hàng hợp lệ đã mua (càng cao càng tốt).
- **Monetary ($M$)**: Tổng số tiền khách hàng đã chi tiêu (càng cao càng tốt).

Mục tiêu của notebook này là triển khai kỹ thuật **Phân cụm Phân cấp Tích lũy (Agglomerative Hierarchical Clustering)**:
1. **Khám phá cấu trúc cụm từ dưới lên (Bottom-up)**.
2. **Thử nghiệm các tiêu chí liên kết (Linkage Criteria)**: `ward`, `complete`, `average` (và `single`).
3. **Thử nghiệm dải số cụm $k \in [2, 10]$** và đánh giá qua **Silhouette Score**, **Davies-Bouldin Index**, **Calinski-Harabasz Index**.
4. **Vẽ cây phả hệ Dendrogram**, xác định điểm cắt ngang (Horizontal Cut Line) và phân tích khoảng nhảy khoảng cách (Distance Gap).
5. **Xây dựng Chân dung Khách hàng (Customer Persona Profiling)** và đề xuất hành động marketing thực chiến.
6. **Đánh giá ngoài thời gian (Out-of-time Temporal Evaluation)** trên cửa sổ 90 ngày tương lai.

In [ ]:
# Cài đặt và nạp các thư viện cần thiết
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)

# Thiết lập đường dẫn gốc dự án và style đồ họa
ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from retail_segmentation.pipeline import FEATURE_COLUMNS, RFM_COLUMNS
from retail_segmentation.temporal_evaluation import load_phase

sns.set_theme(style='whitegrid', font='sans-serif')
plt.rcParams.update({'figure.dpi': 150, 'figure.autolayout': True})
print('✓ Đã nạp thành công các thư viện phân tích!')

## 1. Nạp và Khám Phá Dữ Liệu RFM Chuẩn Hóa

Chúng ta sử dụng tập dữ liệu giai đoạn **Development Phase** (trước ngày 01/09/2011, gồm **2.716 khách hàng**).
Dữ liệu đầu vào của mô hình là ma trận $X$ gồm 3 đặc trưng đã qua phép biến đổi logarit tự nhiên `log1p` và chuẩn hóa z-score `StandardScaler`:
- `Recency_scaled`
- `Frequency_scaled`
- `Monetary_scaled`

> **Lưu ý**: Việc log-transform và chuẩn hóa Standard Scaling là điều kiện tiên quyết bắt buộc trước khi tính khoảng cách Euclidean, vì phân phối RFM trong bán lẻ thường bị lệch phải (right-skewed) rất nặng với sự xuất hiện của các 'cá voi' (Whales/VIPs).

In [ ]:
# Tải dữ liệu giai đoạn development
rfm_dev, feat_dev, outcomes_dev = load_phase(ROOT / 'data_cleaned', 'development')
X_dev = feat_dev[FEATURE_COLUMNS].to_numpy()

print(f'Tổng số khách hàng phân tích (Development Phase): {len(rfm_dev):,}')
print('Ma trận đặc trưng huấn luyện X shape:', X_dev.shape)

display(feat_dev.head())
display(rfm_dev.describe())

## 2. Phân Tích Cây Phân Cấp (Dendrogram) & So Sánh Các Linkage

### Nguyên Lý Phân Cụm Phân Cấp Tích Lũy (Agglomerative)
- **Khởi tạo**: Ban đầu, mỗi khách hàng là 1 cụm độc lập ($N$ cụm với $N=2.716$).
- **Lặp lại**: Ở mỗi bước, tính khoảng cách giữa tất cả các cặp cụm và gộp 2 cụm có khoảng cách nhỏ nhất lại thành một cụm mới.
- **Kết thúc**: Sau $N-1$ bước gộp, toàn bộ khách hàng hội tụ về 1 cụm duy nhất.

### 4 Tiêu Chí Liên Kết (Linkage Methods):
1. **Ward Linkage (`ward`)**: Tối thiểu hóa mức tăng tổng phương sai bên trong các cụm được gộp (Variance / Error Sum of Squares $\Delta ESS$). Ưu tiên tạo các cụm hình cầu lồi (spherical) có kích thước cân đối.
2. **Complete Linkage (`complete`)**: Khoảng cách giữa 2 cụm là khoảng cách xa nhất giữa 2 phần tử bất kỳ ($D(A, B) = \max \{d(x, y): x \in A, y \in B\}$). Tránh chuỗi dài, tạo các cụm chặt chẽ.
3. **Average Linkage (`average`)**: Khoảng cách trung bình giữa tất cả các cặp phần tử của 2 cụm. Cân bằng giữa Ward và Complete.
4. **Single Linkage (`single`)**: Khoảng cách gần nhất giữa 2 cụm. Thường bị hiện tượng xâu chuỗi (Chaining Effect) - gom một dải dài các điểm vào 1 cụm lớn.

In [ ]:
# Tính ma trận liên kết cho cả 4 phương pháp
print('Đang tính toán ma trận liên kết Linkage Matrices...')
Z_ward = linkage(X_dev, method='ward', metric='euclidean')
Z_complete = linkage(X_dev, method='complete', metric='euclidean')
Z_average = linkage(X_dev, method='average', metric='euclidean')
Z_single = linkage(X_dev, method='single', metric='euclidean')

# 1. Vẽ so sánh 4 Dendrogram (rút gọn hiển thị 30 nút gộp cuối cùng)
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()
methods = [('Ward', Z_ward), ('Complete', Z_complete), ('Average', Z_average), ('Single', Z_single)]

for idx, (m_name, Z) in enumerate(methods):
    dendrogram(
        Z,
        truncate_mode='lastp',
        p=30,
        leaf_rotation=90,
        leaf_font_size=9,
        show_contracted=True,
        ax=axes[idx],
    )
    axes[idx].set_title(f'Linkage: {m_name}', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Merger Distance (Khoảng cách gộp)')
    axes[idx].set_xlabel('Số lượng phần tử trong cụm rút gọn')

plt.suptitle('So Sánh Cây Phả Hệ Dendrogram Giữa 4 Phương Pháp Linkage', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Phân Tích Chi Tiết Cây Phả Hệ Ward & Đường Cắt Ngang (Horizontal Cut Line)

Trong Dendrogram của **Ward Linkage**, trục tung biểu diễn mức gia tăng phương sai (Ward Distance).
- Một số cụm $k$ tự nhiên và ổn định thường tương ứng với một **khoảng nhảy khoảng cách lớn (Large Merger Distance Gap)** trên trục tung mà không có nhánh gộp ngang nào cắt qua.
- Quan sát thấy khoảng cách từ $\sim 31$ đến $\sim 45$ là một khoảng trống rất rõ nét, dẫn đến 2 điểm cắt sáng giá:
  - Cắt tại ngưỡng $\sim 38.0$ cho ra **$k = 4$ cụm**.
  - Cắt tại ngưỡng $\sim 31.0$ cho ra **$k = 5$ cụm**.

In [ ]:
# Vẽ Dendrogram Ward với đường cắt ngang lựa chọn số cụm
plt.figure(figsize=(13, 7))
ddata = dendrogram(
    Z_ward,
    truncate_mode='lastp',
    p=35,
    leaf_rotation=90,
    leaf_font_size=10,
    show_contracted=True,
    color_threshold=38.0,
)

plt.axhline(y=38.0, color='crimson', linestyle='--', linewidth=2, label='Đường cắt k=4 (Ngưỡng khoảng cách = 38.0)')
plt.axhline(y=31.0, color='darkorange', linestyle=':', linewidth=2, label='Đường cắt k=5 (Ngưỡng khoảng cách = 31.0)')

plt.title('Dendrogram Ward Linkage Với Các Đường Cắt Ngang Lựa Chọn Số Cụm k', fontsize=14, fontweight='bold')
plt.xlabel('Các nhánh phân cụm (Số trong ngoặc là quy mô số khách hàng của nút)')
plt.ylabel('Ward Merger Distance (Mức gia tăng phương sai ΔESS)')
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

## 3. Thử Nghiệm Grid Search: Đánh Giá Đa Tiêu Chí với $k \in [2, 10]$

Để chọn lựa khách quan, chúng ta tiến hành khảo sát toàn diện qua 3 phương pháp liên kết (`ward`, `complete`, `average`) và số lượng cụm $k$ từ 2 đến 10 dựa trên 3 thước đo chuẩn nội suy (Internal Validation Metrics):
1. **Silhouette Score**: Đo lường sự tương đồng của một điểm với cụm của chính nó so với cụm gần nhất khác (Phạm vi $[-1, 1]$, càng gần 1 càng phân tách rõ nét).
2. **Davies-Bouldin Index**: Tỷ số giữa độ phân tán nội cụm và khoảng cách liên cụm (Càng thấp càng tốt, $\ge 0$).
3. **Calinski-Harabasz Index**: Tỷ số giữa phương sai liên cụm và phương sai nội cụm (Càng cao càng tốt).

In [ ]:
# Grid Search trên các liên kết và số cụm k
linkages = ['ward', 'complete', 'average']
k_range = range(2, 11)
records = []

for method in linkages:
    for k in k_range:
        t0 = time.perf_counter()
        model = AgglomerativeClustering(n_clusters=k, linkage=method, metric='euclidean')
        labels = model.fit_predict(X_dev)
        runtime = time.perf_counter() - t0

        _, counts = np.unique(labels, return_counts=True)
        records.append({
            'linkage': method,
            'k': k,
            'silhouette': round(float(silhouette_score(X_dev, labels)), 4),
            'davies_bouldin': round(float(davies_bouldin_score(X_dev, labels)), 4),
            'calinski_harabasz': round(float(calinski_harabasz_score(X_dev, labels)), 2),
            'min_cluster_size': int(np.min(counts)),
            'max_cluster_size': int(np.max(counts)),
            'runtime_seconds': round(runtime, 4),
        })

eval_df = pd.DataFrame(records)
display(eval_df.pivot(index='k', columns='linkage', values=['silhouette', 'davies_bouldin', 'calinski_harabasz']))

In [ ]:
# Vẽ biểu đồ so sánh 3 chỉ số giữa các Linkage theo k
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = {'ward': '#1f77b4', 'complete': '#2ca02c', 'average': '#ff7f0e'}
markers = {'ward': 'o', 'complete': 's', 'average': '^'}

# Silhouette
for m in linkages:
    sub = eval_df[eval_df['linkage'] == m]
    axes[0].plot(sub['k'], sub['silhouette'], marker=markers[m], label=m.capitalize(), color=palette[m])
axes[0].set_title('Silhouette Score (Cao hơn là tốt hơn)', fontweight='bold')
axes[0].set_xlabel('Số lượng cụm (k)')
axes[0].set_ylabel('Silhouette')
axes[0].legend()

# Davies-Bouldin
for m in linkages:
    sub = eval_df[eval_df['linkage'] == m]
    axes[1].plot(sub['k'], sub['davies_bouldin'], marker=markers[m], label=m.capitalize(), color=palette[m])
axes[1].set_title('Davies-Bouldin Index (Thấp hơn là tốt hơn)', fontweight='bold')
axes[1].set_xlabel('Số lượng cụm (k)')
axes[1].set_ylabel('Davies-Bouldin')
axes[1].legend()

# Calinski-Harabasz
for m in linkages:
    sub = eval_df[eval_df['linkage'] == m]
    axes[2].plot(sub['k'], sub['calinski_harabasz'], marker=markers[m], label=m.capitalize(), color=palette[m])
axes[2].set_title('Calinski-Harabasz Index (Cao hơn là tốt hơn)', fontweight='bold')
axes[2].set_xlabel('Số lượng cụm (k)')
axes[2].set_ylabel('Calinski-Harabasz')
axes[2].legend()

plt.suptitle('So Sánh Hiệu Quả Các Phương Pháp Linkage Theo Số Cụm k', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### 💡 Phân Tích Chuyên Sâu Kết Quả Grid Search

1. **Tại sao `ward` vượt trội hoàn toàn so với `complete` và `average`?**
   - **Calinski-Harabasz của Ward cực cao** (1.656 tại $k=4$), gấp hơn **3 lần** Complete (584) và gấp hơn **13 lần** Average (125). Điều này chứng minh các cụm do Ward tạo ra vô cùng chặt chẽ và phân tách rõ rệt trong không gian Euclidean.
   - **Sự cân bằng kích thước cụm (Cluster Balance)**: Ward phân bổ các cụm rất hợp lý (ở $k=4$, cụm nhỏ nhất là 215 khách hàng và cụm lớn nhất là 1.202 khách hàng). Ngược lại, `average` và `complete` bị hiện tượng cô lập điểm ngoại lai (Outlier Splitting), tạo ra những cụm chỉ có 1 đến 4 khách hàng, hoàn toàn mất đi ý nghĩa kinh doanh phân khúc.

2. **Tại sao chọn $k = 4$?**
   - Tại $k=4$, Silhouette của Ward duy trì mức tốt ($0.3089$), Davies-Bouldin đạt $1.044$, và Calinski-Harabasz đạt $1.656$.
   - Phù hợp hoàn hảo với **4 nhóm hành vi kinh điển trong bán lẻ**: *Champions/VIPs*, *Loyal Customers*, *Potential/Promising*, và *At-Risk/Lost Customers*.
   - Dendrogram thể hiện khoảng nhảy phương sai rất rõ rệt ở mức cắt này.

In [ ]:
# Huấn luyện mô hình tối ưu Ward với k=4
optimal_k = 4
optimal_linkage = 'ward'

model = AgglomerativeClustering(n_clusters=optimal_k, linkage=optimal_linkage, metric='euclidean')
labels_dev = model.fit_predict(X_dev)

# Kết nối nhãn phân cụm với dữ liệu RFM gốc
rfm_segmented = rfm_dev.copy()
rfm_segmented['cluster'] = labels_dev
print(f'Đã hoàn thành phân cụm cho {len(rfm_segmented):,} khách hàng với k={optimal_k}!')

## 4. Chân Dung Khách Hàng (Customer Persona Profiling)

Để chuyển đổi kết quả học máy thành giá trị kinh doanh, chúng ta tính toán các chỉ số thống kê trung vị (Median) và trung bình (Mean) của các giá trị RFM thực tế (chưa scale) cho từng nhóm.

In [ ]:
# Tính toán thống kê mô tả RFM theo cụm
summary = rfm_segmented.groupby('cluster').agg(
    customers=('CustomerID', 'count'),
    recency_median=('Recency', 'median'),
    frequency_median=('Frequency', 'median'),
    monetary_median=('Monetary', 'median'),
    monetary_total=('Monetary', 'sum'),
).reset_index()

summary['customer_share'] = summary['customers'] / len(rfm_segmented)
summary['revenue_share'] = summary['monetary_total'] / rfm_segmented['Monetary'].sum()

# Đặt tên ngữ nghĩa kinh doanh cho từng phân khúc
persona_names = {
    0: 'Loyal Customers (Mua Thường Xuyên, Giá Trị Khá)',
    1: 'At-Risk / Lost (Lâu Không Mua, Giá Trị Thấp)',
    2: 'Champions / VIP (Chi Tiêu Khủng, Tần Suất Cực Cao)',
    3: 'Potential / Promising (Mới Mua Gần Đây, Đang Phát Triển)',
}
summary['persona'] = summary['cluster'].map(persona_names)
rfm_segmented['persona'] = rfm_segmented['cluster'].map(persona_names)

display(summary[['cluster', 'persona', 'customers', 'customer_share', 'recency_median', 'frequency_median', 'monetary_median', 'revenue_share']])

In [ ]:
# Trực quan hóa phân khúc bằng PCA 2D và Boxplots
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_dev)
rfm_segmented['PCA1'] = X_pca[:, 0]
rfm_segmented['PCA2'] = X_pca[:, 1]

# 1. Bản đồ phân cụm trên không gian PCA 2D
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm_segmented,
    x='PCA1',
    y='PCA2',
    hue='persona',
    palette='tab10',
    alpha=0.75,
    s=45,
)
plt.title(f'Bản Đồ Phân Cụm Khách Hàng (PCA 2D - Giải thích {pca.explained_variance_ratio_.sum()*100:.1f}% Phương Sai)', fontsize=13, fontweight='bold')
plt.xlabel(f'Thành phần chính 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'Thành phần chính 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

# 2. Phân bố RFM qua Boxplots (thang đo log cho Frequency và Monetary)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=rfm_segmented, x='cluster', y='Recency', ax=axes[0], hue='cluster', palette='Set2', legend=False)
axes[0].set_title('Recency (Số ngày kể từ lần mua cuối - Thấp là tốt)')
axes[0].set_ylabel('Ngày')

sns.boxplot(data=rfm_segmented, x='cluster', y='Frequency', ax=axes[1], hue='cluster', palette='Set2', legend=False)
axes[1].set_yscale('log')
axes[1].set_title('Frequency (Số lượng hóa đơn - Thang log)')
axes[1].set_ylabel('Số đơn hàng')

sns.boxplot(data=rfm_segmented, x='cluster', y='Monetary', ax=axes[2], hue='cluster', palette='Set2', legend=False)
axes[2].set_yscale('log')
axes[2].set_title('Monetary (Doanh thu đã chi tiêu £ - Thang log)')
axes[2].set_ylabel('Tổng tiền chi tiêu (£)')

plt.suptitle('Phân Phối Đặc Trưng RFM Theo Từng Cụm Khách Hàng', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 5. Đánh Giá Khả Năng Dự Báo Tương Lai (Out-of-Time Temporal Evaluation)

Một mô hình phân cụm tốt không chỉ có chỉ số nội suy đẹp, mà phải **phản ánh đúng hành vi mua sắm trong tương lai**.
Chúng ta kiểm tra kết quả hành vi trong **90 ngày tiếp theo** của từng cụm dựa trên dữ liệu thực tế:

In [ ]:
# Tải kết quả đánh giá ngoài thời gian từ thư mục reports
future_path = ROOT / 'reports' / 'temporal' / 'development' / 'future_cluster_profiles.csv'
if future_path.exists():
    future_df = pd.read_csv(future_path)
    future_df['persona'] = future_df['cluster'].map(persona_names)
    display(future_df[['cluster', 'persona', 'customers', 'repurchased', 'repurchase_rate', 'future_revenue', 'future_revenue_share']])
    
    # Vẽ biểu đồ tỷ lệ mua lại và doanh thu tương lai
    fig, ax1 = plt.subplots(figsize=(10, 5))
    color = '#1f77b4'
    ax1.set_xlabel('Cụm Khách Hàng', fontweight='bold')
    ax1.set_ylabel('Tỷ Lệ Mua Lại 90 Ngày Tới (Repurchase Rate)', color=color, fontweight='bold')
    bars = ax1.bar(future_df['cluster'].astype(str), future_df['repurchase_rate'], color=color, alpha=0.6, width=0.4, align='center', label='Tỷ lệ mua lại')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_ylim(0, 1.1)
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02, f'{height*100:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    ax2 = ax1.twinx()
    color = '#d62728'
    ax2.set_ylabel('Đóng Góp Doanh Thu Tương Lai (£)', color=color, fontweight='bold')
    line = ax2.plot(future_df['cluster'].astype(str), future_df['future_revenue'], color=color, marker='o', linewidth=2.5, label='Doanh thu tương lai')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title('Hành Vi Mua Sắm 90 Ngày Tương Lai Theo Từng Phân Khúc Agglomerative', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Chưa tìm thấy future_cluster_profiles.csv. Hãy chạy scripts/evaluate_temporal.py trước!')

## 6. Đề Xuất Chiến Lược Marketing & Chăm Sóc Khách Hàng (Actionable Playbook)

| Cụm | Nhóm Khách Hàng | Đặc Điểm RFM | Hành Vi 90 Ngày Tới | Chiến Lược Marketing Khuyên Dùng |
| :---: | :--- | :--- | :--- | :--- |
| **Cụm 2** | **Champions / VIP** | $R=9$ ngày, $F=9$ đơn, $M=3.450£$ | **94.9% mua lại**, tạo ra **54.7% doanh thu** | **Chăm sóc đặc quyền VIP**: Quản lý tài khoản riêng (Dedicated account manager), quyền trải nghiệm sản phẩm mới trước, quà tri ân ngày lễ, tuyệt đối không spam giảm giá đại trà. |
| **Cụm 0** | **Loyal Customers** | $R=27$ ngày, $F=3$ đơn, $M=924£$ | **64.6% mua lại**, tạo ra **28.6% doanh thu** | **Chương trình khách hàng thân thiết (Loyalty Program)**: Tích điểm đổi quà, gợi ý combo sản phẩm (Cross-sell/Up-sell) dựa trên lịch sử mua sắm để đẩy họ thành VIP. |
| **Cụm 3** | **Potential / Promising** | $R=20$ ngày, $F=1$ đơn, $M=304£$ | **41.1% mua lại**, tạo ra **3.4% doanh thu** | **Kích hoạt đơn hàng thứ 2 (Onboarding & Nurturing)**: Gửi voucher chào mừng cho lần mua tiếp theo trong vòng 30 ngày, khảo sát mức độ hài lòng về sản phẩm đầu tiên. |
| **Cụm 1** | **At-Risk / Lost** | $R=97$ ngày, $F=1$ đơn, $M=261£$ | Chỉ **30.7% mua lại**, doanh thu chỉ **168£/người** | **Chiến dịch tái kích hoạt (Win-back Campaign)**: Gửi email 'Chúng tôi nhớ bạn' kèm ưu đãi giảm giá sâu có giới hạn thời gian (flash sale) để kéo họ quay lại; nếu không phản hồi thì loại khỏi danh sách email marketing thường xuyên để tiết kiệm chi phí gửi mail. |